# Generate and validate `miiv` configs

This notebook generates WeavEHR extraction and concept configurations from the RICU JSON dictionaries. Existing WeavEHR configs are supplied as the canonical reference: matching extraction files are copied exactly, while missing tables are generated conservatively and marked `review_required`.


In [ ]:
from pathlib import Path
from weavehr_ricu_converter import generate_project

REPO = Path.cwd().resolve().parents[1]
INPUT = REPO / 'examples' / 'input'
OUTPUT = REPO / 'generated' / 'miiv'


In [ ]:
result = generate_project(
    concept_dict=INPUT / 'concept-dict.json',
    data_sources=INPUT / 'data-sources.json',
    output_config=OUTPUT,
    sources=['miiv'],
    settings=REPO / 'examples' / 'settings.yml',
    overwrite=True,
)
result


## Inspect generated files


In [ ]:
import yaml
manifest = yaml.safe_load((OUTPUT / 'generation_manifest.yml').read_text())
manifest


In [ ]:
heuristic = []
for path in OUTPUT.rglob('*.yml'):
    text = path.read_text()
    if 'review_required: true' in text:
        heuristic.append(path.relative_to(OUTPUT))
heuristic[:50], len(heuristic)


## Exact reference comparison
Files originating from an existing WeavEHR configuration should be semantically identical. The comparison below ignores YAML formatting and compares parsed objects.


In [ ]:
# Adapt the reference dataset/version path when necessary.
# The converter itself resolves common aliases such as miiv -> mimic-iv.
generated_tables = list((OUTPUT / 'datasets' / 'miiv' / 'generated' / 'tables').glob('*.yml'))
len(generated_tables)
